# 04 · ESM1b — a protein language model

ESM1b (Brandes et al. 2023, *Nat Genet*, PMID 37563329) is an *LLM for protein sequences*. Like EVE it is **unsupervised**, but it scores variants on a **log-likelihood ratio (LLR)** whose scale runs *backwards* from every other tool here.

> ### The 3 CFTR UniProt IDs (P13569, P13569-2, P13569-3)
> ESM1b variant files (ntranoslab) list **three** CFTR-related isoforms:
> **P13569** is the **canonical** CFTR isoform (1480 aa; matches MANE `NM_000492.4`),
> while **-2** and **-3** are alternative UniProt isoforms (differ by alternative
> splicing). **Use the canonical `P13569`** so residue numbering lines up with
> AlphaMissense / CFTR2 / gnomAD — picking `-2`/`-3` silently shifts positions and
> breaks the `protein_variant` join. *(Isoform details: UniProt P13569.)*

> ✅ **REAL DATA.** Full CFTR **saturation** ESM1b LLR — **~28,120** variants (all 1,480 residues), `data/esm1b_cftr.csv`, built by `build_esm1b.py` from the ntranoslab release (canonical UniProt **P13569**). `source == 'REAL'`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 2 · ESM1b — a protein language model

**What is it?** ESM1b is a **protein language model** — think of it as an *LLM for protein sequences*. Just as a text LLM learns which word is likely to come next, ESM1b was trained on millions of natural protein sequences to predict which amino acid "belongs" at each position given its neighbours. It never saw the alignment explicitly; it learned protein grammar directly from raw sequences.

To score a variant, ESM1b compares how *likely* the model thinks the **mutant** amino acid is versus the **wild-type** amino acid at that position. This is a **log-likelihood ratio (LLR)**:

$$\text{LLR} = \log \frac{P(\text{mutant amino acid})}{P(\text{wild-type amino acid})}$$

> ### 🔄 IMPORTANT — ESM1b runs *backwards* from the other tools
> A **more negative** LLR means the model finds the mutation more *surprising* → **more damaging**. This is the **opposite direction** to EVE, AlphaMissense, and REVEL, where *higher* = worse.
> 
> - **Cut-off:** `LLR <= -7.5` ~ pathogenic. **Lower (more negative) = worse.**
> - Unsupervised (learned from sequences only) → low circularity, just like EVE.

> ### 📥 How to get REAL ESM1b data
> Bulk, per-protein ESM1b variant-effect files are published as the **Brandes et al. 2023 supplement** and on **HuggingFace**. Download the CFTR file and join on `protein_variant`.

In [2]:
esm = tk.load_esm1b()      # REAL — full CFTR saturation LLR (~28,120)
print(f"{len(esm):,} REAL ESM1b variants | source: {esm['source'].unique().tolist()}")
print('LLR range:', esm['esm1b_score'].min(), '->', esm['esm1b_score'].max(),
      '| pathogenic (<= -7.5):', int((esm['esm1b_score'] <= -7.5).sum()))
esm.head(8)

28,120 REAL ESM1b variants | source: ['REAL']
LLR range: -23.793 -> 4.943 | pathogenic (<= -7.5): 14948


,protein_variant,esm1b_score,source
0,M1K,-5.762,REAL
1,M1R,-6.544,REAL
2,M1H,-8.114,REAL
3,M1E,-6.197,REAL
4,M1D,-6.949,REAL
5,M1N,-6.897,REAL
6,M1Q,-7.112,REAL
7,M1T,-6.903,REAL


## 2 · Turn an ESM1b LLR into a call

`tk.call_from_score(score, 'esm1b')` bakes in the **direction**: cut at `<= -7.5`, and **lower / more negative = worse**. You do not juggle the sign yourself.

In [3]:
esm['esm1b_call'] = esm['esm1b_score'].apply(lambda s: tk.call_from_score(s, 'esm1b'))
esm[['protein_variant', 'esm1b_score', 'esm1b_call', 'source']].head(12)

,protein_variant,esm1b_score,esm1b_call,source
0,M1K,-5.762,benign,REAL
1,M1R,-6.544,benign,REAL
2,M1H,-8.114,pathogenic,REAL
3,M1E,-6.197,benign,REAL
4,M1D,-6.949,benign,REAL
5,M1N,-6.897,benign,REAL
6,M1Q,-7.112,benign,REAL
7,M1T,-6.903,benign,REAL
8,M1S,-6.292,benign,REAL
9,M1C,-8.059,pathogenic,REAL


### The sign flip, made concrete

It is worth staring at *one* variant until the backwards scale clicks. Run the next cell.

In [4]:
# Most negative ESM1b = most damaging.
row = esm.sort_values('esm1b_score').iloc[0]
print(f"Variant {row['protein_variant']}:")
print(f"  ESM1b = {row['esm1b_score']:>6}  ->  {row['esm1b_call']:<10} "
      f"(LOW / negative score, cut at -7.5, so low = pathogenic)")
print('A more negative LLR = a more surprising mutation = more damaging —')
print('the opposite direction to EVE, AlphaMissense and REVEL.')

Variant R289P:
  ESM1b = -23.793  ->  pathogenic (LOW / negative score, cut at -7.5, so low = pathogenic)
A more negative LLR = a more surprising mutation = more damaging —
the opposite direction to EVE, AlphaMissense and REVEL.


## Key takeaways

1. **ESM1b** scores the mutant-vs-wild-type **log-likelihood ratio**; cut `<= -7.5` ~ pathogenic — **lower / more negative = worse** (opposite to EVE). `tk.call_from_score` handles the sign.
2. **Unsupervised** → low circularity vs ClinVar (like EVE).
3. This notebook now uses **REAL ESM1b** — full CFTR saturation (~28,120 variants, P13569), `source == REAL`.

**Next:** notebook 05 — **REVEL** (supervised → circularity).